# 05 — RAG with LCEL

## End-to-End Retrieval-Augmented Generation Pipeline

**Experiment ID:** RAG-001

### Objective

Build the first complete RAG pipeline using the decisions established in Notebooks 02–04.

### Selected Baseline

```text
Dataset           : Pharma Sales CSV
Documents         : 300
Chunk Size        : 500
Chunk Overlap     : 50
Chunks            : 1,538
Embedding Model   : text-embedding-3-large
Retriever         : Similarity Search
k                 : 3
LLM               : gpt-4.1-mini
Temperature       : 0
```

### Pipeline

```text
User Question
      ↓
Retriever
      ↓
Top 3 Relevant Chunks
      ↓
Context Construction
      ↓
Grounded Prompt
      ↓
GPT-4.1-mini
      ↓
Answer + Source Metadata
```

## 1. Environment & Imports

The implementation intentionally keeps the pipeline simple and explainable.

Main components:

- `Chroma` — vector store
- `OpenAIEmbeddings` — embeddings
- `ChatOpenAI` — generation
- `ChatPromptTemplate` — grounded prompt
- `RunnablePassthrough` — preserves the question
- `StrOutputParser` — converts the model response to text

In [25]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY environment variable is not set. "
        "Please configure it in your .env file."
    )

print("Environment configured successfully.")

Environment configured successfully.


C:\Users\visha\AppData\Local\Temp\ipykernel_46380\2838299649.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


## 2. Define the Project Baseline

These values come directly from the previous experiments.

Keeping them in one configuration block makes the final RAG pipeline easy to understand and modify.

In [26]:
RAG_CONFIG = {
    "data_path": "../data/Pharma_Sales_Long.csv",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "embedding_model": "text-embedding-3-large",
    "retrieval_k": 3,
    "llm_model": "gpt-4.1-mini",
    "temperature": 0,
}

for key, value in RAG_CONFIG.items():
    print(f"{key}: {value}")

data_path: ../data/Pharma_Sales_Long.csv
chunk_size: 500
chunk_overlap: 50
embedding_model: text-embedding-3-large
retrieval_k: 3
llm_model: gpt-4.1-mini
temperature: 0


## 3. Load the Source Documents

The canonical dataset is loaded independently so this notebook is reproducible.

Expected:

```text
300 CSV rows → 300 LangChain Documents
```

In [27]:
loader = CSVLoader(
    file_path=RAG_CONFIG["data_path"],
    encoding="utf-8"
)

data = loader.load()

assert len(data) == 300, (
    f"Expected 300 documents, found {len(data)}."
)

print(f"Loaded {len(data)} documents.")

Loaded 300 documents.


## 4. Recreate the Selected Chunking Strategy

Selected in Notebook 02:

```text
chunk_size    = 500
chunk_overlap = 50
```

Expected:

```text
1,538 chunks
```

In [28]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=RAG_CONFIG["chunk_size"],
    chunk_overlap=RAG_CONFIG["chunk_overlap"],
)

chunks = text_splitter.split_documents(data)

assert len(chunks) == 1538, (
    f"Expected 1,538 chunks, found {len(chunks)}."
)

print(f"Documents : {len(data)}")
print(f"Chunks    : {len(chunks)}")

Documents : 300
Chunks    : 1538


## 5. Create the Vector Store

Notebook 03 selected `text-embedding-3-large`.

We create a Chroma vector store from the same 1,538 chunks.

In [29]:
embedding_model = OpenAIEmbeddings(
    model=RAG_CONFIG["embedding_model"]
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="rag_lcel_baseline",
)

print("✓ Chroma vector store created.")
print(f"✓ Stored chunks: {len(chunks)}")

✓ Chroma vector store created.
✓ Stored chunks: 1538


## 6. Create the Retriever

Notebook 04 selected `k=3` as the initial retrieval baseline.

In [30]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": RAG_CONFIG["retrieval_k"]
    }
)

print(
    f"Retriever configured with "
    f"k={RAG_CONFIG['retrieval_k']}"
)

Retriever configured with k=3


## 7. Test Retrieval Before Connecting the LLM

Before building the complete RAG chain, verify retrieval independently.

```text
Retrieval problem ≠ Generation problem
```

If retrieval returns poor context, changing the prompt or LLM will not fix the underlying retrieval problem.

In [31]:
test_question = (
    "Which sales records describe WELIREG discussions related to renal cell "
    "carcinoma in territories where customer engagement or follow-up activity "
    "was also mentioned?"
)

retrieved_docs = retriever.invoke(test_question)

print(
    f"Retrieved {len(retrieved_docs)} documents "
    f"for the test question."
)

for rank, document in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Source row: {document.metadata.get('row')}")
    print(f"Source: {document.metadata.get('source')}")
    print("Content:")
    print(document.page_content)

Retrieved 3 documents for the test question.
Rank: 1
Source row: 16
Source: ../data/Pharma_Sales_Long.csv
Content:
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,
Rank: 2
Source row: 85
Source: ../data/Pharma_Sales_Long.csv
Content:
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up p

## 8. Build the Context Formatter

The retriever returns LangChain `Document` objects.

The LLM needs a readable context string.

We convert:

```text
List[Document]
      ↓
Context String
```

while retaining source row information for inspection.

In [32]:
def format_docs(documents):
    formatted_documents = []

    for rank, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "Unknown")
        row = document.metadata.get("row", "Unknown")

        formatted_documents.append(
            f"[Source {rank} | Row {row} | File {source}]\n"
            f"{document.page_content}"
        )

    return "\n\n".join(formatted_documents)


context = format_docs(retrieved_docs)

print(context)

[Source 1 | Row 16 | File ../data/Pharma_Sales_Long.csv]
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,

[Source 2 | Row 85 | File ../data/Pharma_Sales_Long.csv]
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promoti

## 9. Create a Grounded RAG Prompt

The prompt establishes the main grounding rule:

> Answer using only the retrieved context.

If the context does not contain enough information, the model must use the configured fallback response.

In [33]:
prompt = ChatPromptTemplate.from_template(
    '''
You are a helpful enterprise data assistant.

Answer the user's question using ONLY the provided context.

Rules:
1. Do not use information that is not present in the context.
2. Do not invent facts, numbers, products, territories, or business conclusions.
3. If the context does not contain enough information to answer the question,
   say exactly:
   "I don't know based on the provided context."
4. Keep the answer concise and directly relevant to the question.
5. When useful, mention the source row(s) that support the answer.

Context:
{context}

Question:
{question}

Answer:
'''
)

print("Grounded RAG prompt created.")

Grounded RAG prompt created.


## 10. Initialize the LLM

The LLM is used for generation after retrieval.

`temperature=0` is intentional because we want consistent and factual behavior rather than creative generation.

In [34]:
llm = ChatOpenAI(
    model=RAG_CONFIG["llm_model"],
    temperature=RAG_CONFIG["temperature"],
)

print(
    f"LLM configured: {RAG_CONFIG['llm_model']} "
    f"| temperature={RAG_CONFIG['temperature']}"
)

LLM configured: gpt-4.1-mini | temperature=0


## 11. Build the RAG Chain Using LCEL

The chain performs:

```text
Question
   ↓
Retriever
   ↓
Context Formatter
   ↓
Grounded Prompt
   ↓
LLM
   ↓
Output Parser
   ↓
Answer
```

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(), # "Don't change the input. Just pass it through.(vector database and LLM Prompt), Take the incoming question and create two outputs: context and question."
    }
    | prompt
    | llm
    | StrOutputParser()  # It converts the model output into a normal string.
)

print("✓ RAG LCEL chain created successfully.")

✓ RAG LCEL chain created successfully.


## 12. Execute the First End-to-End RAG Query

In [36]:
response = rag_chain.invoke(test_question)

print("=" * 80)
print("QUESTION")
print("=" * 80)
print(test_question)

print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(response)

QUESTION
Which sales records describe WELIREG discussions related to renal cell carcinoma in territories where customer engagement or follow-up activity was also mentioned?

ANSWER
Sales records in Source 1 (Row 16), Source 2 (Row 85), and Source 3 (Row 227) describe WELIREG discussions related to renal cell carcinoma that include customer engagement and follow-up activity in the territory insights.


In [46]:
# Testing Chain

question = "What is RAG?"

response = rag_chain.invoke(question)

print("=" * 80)
print("QUESTION")
print("=" * 80)
print(question)

print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(response)

QUESTION
What is RAG?

ANSWER
I don't know based on the provided context.


## 13. Keep Retrieval and Generation Observable

For evaluation, retrieval should remain inspectable.

The helper below returns:

- Question
- Answer
- Retrieved Documents
- Formatted Context

In [37]:
def run_rag_with_sources(question):
    documents = retriever.invoke(question)
    context = format_docs(documents)

    answer = llm.invoke(
        prompt.invoke({
            "context": context,
            "question": question,
        })
    )

    return {
        "question": question,
        "answer": answer.content,
        "documents": documents,
        "context": context,
    }


result = run_rag_with_sources(test_question)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for rank, document in enumerate(result["documents"], start=1):
    print(
        f"{rank}. Row={document.metadata.get('row')} "
        f"| Source={document.metadata.get('source')}"
    )

ANSWER:
Sales records in Source 1 (Row 16), Source 2 (Row 85), and Source 3 (Row 227) describe WELIREG discussions related to renal cell carcinoma that include customer engagement and follow-up activity in the territory insights.

SOURCES:
1. Row=16 | Source=../data/Pharma_Sales_Long.csv
2. Row=85 | Source=../data/Pharma_Sales_Long.csv
3. Row=227 | Source=../data/Pharma_Sales_Long.csv


## 14. Define the Four Evaluation Questions

The same questions are reused from Notebooks 03 and 04.

This gives the project a consistent evaluation path:

```text
Embedding Experiment
        ↓
Similarity Search Experiment
        ↓
RAG Generation Experiment
```

In [38]:
evaluation_questions = {
    "Q001": (
        "Which sales records describe WELIREG discussions related to renal cell "
        "carcinoma in territories where customer engagement or follow-up activity "
        "was also mentioned?"
    ),
    "Q002": (
        "Find records where the sales discussion combines physician engagement, "
        "approved clinical information, and follow-up planning for a vaccine product."
    ),
    "Q003": (
        "Which records indicate regional or territory-level business opportunities "
        "while also referring to prescription trends and customer behavior?"
    ),
    "Q004": (
        "Find records where customer discussions include market access or competitor "
        "comparison together with compliant promotional or scientific-literature activities."
    ),
}

for question_id, question in evaluation_questions.items():
    print("=" * 80)
    print(question_id)
    print(question)

Q001
Which sales records describe WELIREG discussions related to renal cell carcinoma in territories where customer engagement or follow-up activity was also mentioned?
Q002
Find records where the sales discussion combines physician engagement, approved clinical information, and follow-up planning for a vaccine product.
Q003
Which records indicate regional or territory-level business opportunities while also referring to prescription trends and customer behavior?
Q004
Find records where customer discussions include market access or competitor comparison together with compliant promotional or scientific-literature activities.


## 15. Run the Complete RAG Pipeline Against All Questions

For each question we capture:

- Generated answer
- Retrieved source rows
- Number of retrieved documents

In [39]:
rag_results = []

for question_id, question in evaluation_questions.items():
    result = run_rag_with_sources(question)

    source_rows = [
        document.metadata.get("row")
        for document in result["documents"]
    ]

    rag_results.append({
        "Experiment_ID": "RAG-001",
        "Question_ID": question_id,
        "Question": question,
        "Answer": result["answer"],
        "Retrieved_Rows": str(source_rows),
        "Retrieved_Document_Count": len(result["documents"]),
    })

rag_results_df = pd.DataFrame(rag_results)

display(
    rag_results_df[
        [
            "Experiment_ID",
            "Question_ID",
            "Retrieved_Document_Count",
            "Retrieved_Rows",
            "Answer",
        ]
    ]
)

,Experiment_ID,Question_ID,Retrieved_Document_Count,Retrieved_Rows,Answer
0,RAG-001,Q001,3,"[16, 85, 227]","Sales records in Source 1 (Row 16), Source 2 (..."
1,RAG-001,Q002,3,"[19, 74, 238]","Records in Source 1 (Row 19), Source 2 (Row 74..."
2,RAG-001,Q003,3,"[71, 160, 299]","Records in Source 1 (Row 71), Source 2 (Row 16..."
3,RAG-001,Q004,3,"[279, 16, 80]","Records in Source 1 (Row 279), Source 2 (Row 1..."


## 16. Inspect Answers with Supporting Context

A generated answer should not be evaluated without checking the retrieved context.

For each question we display:

1. Question
2. Answer
3. Retrieved source rows
4. Retrieved content

In [40]:
for question_id, question in evaluation_questions.items():

    result = run_rag_with_sources(question)

    print("=" * 100)
    print(question_id)
    print("=" * 100)

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nRETRIEVED SOURCES:")

    for rank, document in enumerate(result["documents"], start=1):
        print("-" * 90)
        print(
            f"Rank {rank} | "
            f"Row {document.metadata.get('row')}"
        )
        print(document.page_content)

Q001

QUESTION:
Which sales records describe WELIREG discussions related to renal cell carcinoma in territories where customer engagement or follow-up activity was also mentioned?

ANSWER:
Sales records in Source 1 (Row 16), Source 2 (Row 85), and Source 3 (Row 227) describe WELIREG discussions related to renal cell carcinoma that include customer engagement and follow-up activity in the territory insights.

RETRIEVED SOURCES:
------------------------------------------------------------------------------------------
Rank 1 | Row 16
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included presc

## 17. Add a Simple Grounding Status Check

This is **not** a production hallucination detector.

It is a basic evaluation helper.

The important manual check remains:

> Can the generated answer be supported by the retrieved documents?

In [41]:
def grounding_status(answer):
    fallback_phrase = "I don't know based on the provided context."

    if fallback_phrase.lower() in answer.lower():
        return "INSUFFICIENT_CONTEXT"

    return "ANSWER_GENERATED"


rag_results_df["Grounding_Status"] = (
    rag_results_df["Answer"]
    .apply(grounding_status)
)

display(
    rag_results_df[
        [
            "Question_ID",
            "Grounding_Status",
            "Retrieved_Document_Count",
        ]
    ]
)

,Question_ID,Grounding_Status,Retrieved_Document_Count
0,Q001,ANSWER_GENERATED,3
1,Q002,ANSWER_GENERATED,3
2,Q003,ANSWER_GENERATED,3
3,Q004,ANSWER_GENERATED,3


## 18. Baseline RAG Architecture

```text
                    CSV Dataset
                         │
                         ▼
                  Chunking 500/50
                         │
                         ▼
                1,538 Chunks
                         │
                         ▼
          text-embedding-3-large
                         │
                         ▼
                    Chroma
                         │
                    User Query
                         │
                         ▼
                  Retriever k=3
                         │
                         ▼
                Retrieved Context
                         │
                         ▼
                 Grounded Prompt
                         │
                         ▼
                   GPT-4.1-mini
                         │
                         ▼
                  Final Answer
                  + Source Rows
```

## 19. What We Have Proven

The project now demonstrates all core RAG stages.

### Data Ingestion

```text
CSV → LangChain Documents
```

### Chunking

```text
300 Documents → 1,538 Chunks
```

### Embedding

```text
text-embedding-3-large
```

selected through a controlled comparison.

### Retrieval

```text
k=3
```

selected as the initial retrieval baseline.

### Generation

```text
Retrieved Context + Question
        ↓
Grounded Prompt
        ↓
GPT-4.1-mini
```

### End-to-End RAG

```text
Question → Retrieval → Context → LLM → Answer
```

## 20. Current Limitations

This notebook is a baseline implementation.

1. The dataset contains repeated or similar Notes content.
2. Retrieval quality is evaluated primarily through similarity scores and manual inspection.
3. There is no labeled ground-truth answer dataset yet.
4. Source citations are basic metadata rather than a formal citation framework.
5. The Chroma store is recreated inside the notebook.
6. There is no response latency or token-cost tracking.
7. There is no automated answer-quality evaluation.

### Potential Next-Level Improvements

- Evaluation dataset with expected answers
- Retrieval precision / recall / MRR
- RAG evaluation framework
- Source-aware answer citations
- Metadata filtering
- Hybrid retrieval
- Reranking
- Persistent vector-store management
- Cost and latency monitoring
- API/service layer

## 21. Persist RAG Evaluation Results

In [42]:
RESULTS_DIR = Path("../experiments/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

rag_results_path = RESULTS_DIR / "rag_lcel_results.csv"

rag_results_df.to_csv(
    rag_results_path,
    index=False
)

print(f"RAG evaluation results saved to: {rag_results_path}")

RAG evaluation results saved to: ..\experiments\results\rag_lcel_results.csv


## 22. Update the Central Experiment Log

The final RAG experiment is recorded in:

```text
experiments/experiment_log.csv
```

The update is rerun-safe.

In [43]:
EXPERIMENT_LOG_PATH = "../experiments/experiment_log.csv"

rag_log = pd.DataFrame([
    {
        "Experiment_ID": "RAG-001",
        "Area": "RAG / LCEL",
        "Configuration": (
            "text-embedding-3-large + "
            "similarity search k=3 + "
            "gpt-4.1-mini"
        ),
        "Question": "Q001-Q004",
        "Documents": len(data),
        "Chunks": len(chunks),
        "Top_K": RAG_CONFIG["retrieval_k"],
        "Results": "4 RAG answers generated",
        "Observation": (
            "End-to-end grounded RAG pipeline evaluated across four "
            "multi-criteria business questions."
        ),
        "Conclusion": (
            "Baseline RAG pipeline successfully connects retrieval, "
            "context construction, grounded prompting and LLM generation."
        ),
    }
])

log_columns = [
    "Experiment_ID",
    "Area",
    "Configuration",
    "Question",
    "Documents",
    "Chunks",
    "Top_K",
    "Results",
    "Observation",
    "Conclusion",
]

rag_log = rag_log[log_columns]

if (
    os.path.exists(EXPERIMENT_LOG_PATH)
    and os.path.getsize(EXPERIMENT_LOG_PATH) > 0
):
    existing_log = pd.read_csv(EXPERIMENT_LOG_PATH)

    if "Experiment_ID" in existing_log.columns:
        existing_log = existing_log[
            existing_log["Experiment_ID"] != "RAG-001"
        ]

        combined_log = pd.concat(
            [existing_log, rag_log],
            ignore_index=True
        )
    else:
        combined_log = rag_log
else:
    combined_log = rag_log

combined_log.to_csv(
    EXPERIMENT_LOG_PATH,
    index=False
)

print(f"Experiment log updated: {EXPERIMENT_LOG_PATH}")
print(f"Total logged experiments: {len(combined_log)}")

Experiment log updated: ../experiments/experiment_log.csv
Total logged experiments: 10


## 23. Final Validation

The notebook is complete when:

- 300 source documents are loaded.
- 1,538 chunks are created.
- `text-embedding-3-large` is used.
- `k=3` is used.
- The LCEL RAG chain executes successfully.
- All four evaluation questions generate answers.
- Retrieved source documents are available for inspection.
- Results are persisted.

In [44]:
assert len(data) == 300
assert len(chunks) == 1538

assert RAG_CONFIG["embedding_model"] == "text-embedding-3-large"
assert RAG_CONFIG["retrieval_k"] == 3

assert len(evaluation_questions) == 4
assert len(rag_results_df) == 4

assert (
    rag_results_df["Retrieved_Document_Count"]
    == RAG_CONFIG["retrieval_k"]
).all()

assert rag_results_df["Answer"].notna().all()

print("✓ RAG LCEL validation passed.")
print("✓ Documents: 300")
print("✓ Chunks: 1,538")
print("✓ Embedding: text-embedding-3-large")
print("✓ Retrieval k: 3")
print("✓ Questions evaluated: 4")
print("✓ Answers generated: 4")
print("✓ Results persisted.")

✓ RAG LCEL validation passed.
✓ Documents: 300
✓ Chunks: 1,538
✓ Embedding: text-embedding-3-large
✓ Retrieval k: 3
✓ Questions evaluated: 4
✓ Answers generated: 4
✓ Results persisted.


# 24. Project Milestone — Baseline RAG Complete

The experimentation phase has now produced a complete baseline RAG system.

```text
                 EXPERIMENTATION
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
     Chunking       Embedding      Retrieval
     500 / 50       3-large          k=3
        │              │              │
        └──────────────┼──────────────┘
                       ▼
                  BASELINE RAG
                       │
                       ▼
                 LCEL PIPELINE
                       │
                       ▼
             Grounded LLM Answer
```

### Current Baseline

```text
Chunking      : 500 / 50
Embedding     : text-embedding-3-large
Vector Store  : Chroma
Retriever     : Similarity Search
k             : 3
LLM           : gpt-4.1-mini
Temperature   : 0
Framework     : LangChain LCEL
```

### Next Engineering Step

The notebooks have established the experimental baseline.

The reusable components can now be extracted into:

```text
src/
    ingestion/
    processing/
    embeddings/
    retrieval/
    generation/
```

The notebooks remain as experiment records and reproducibility artifacts.